Imports and paths

In [3]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

Read csv files in raw.

In [221]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dtype_overrides: dict[str, type] = {"customer_zip_code_prefix": str}

dfs: dict[str, DataFrame] = {
    f.name: pd.read_csv(f, dtype=dtype_overrides) for f in raw_filepaths
}
# ic(dfs)

In [250]:
# dfs.keys()


def describe_table(current_table: str, current_column: str) -> None:
    df = dfs[current_table]

    print(f"current column: {current_column}")
    print("")
    print(df[current_column].sample(5))
    print("")
    print(f"dtype:  {df[current_column].dtype}")
    print(f"table:  {current_table}")
    print(f"column: {current_column}")
    print("")

    # check if contains null
    has_null = df[current_column].isnull().any()
    print(f"contains NULL:  {has_null}")

    # if has_null:
    #     # filter out null
    #     print("contains null")
    #     df_notnull = df[df[current_column].notnull()]
    #     pass

    if df[current_column].dtype != "object":
        print("not an object")

        df[current_column] = df[current_column].dropna()

        print(df[current_column].describe())

        return

    # check if requires 'n' prefix: nchar, nvarchar
    # n = national, means contains unicode
    is_national: bool = not df[current_column].apply(lambda x: str(x).isascii()).all()

    # check if char or varchar
    entry_lengths = df[current_column].dropna().str.len().unique()

    max_length = entry_lengths.max()
    is_fixed_length = entry_lengths.size == 1

    # print(f'is char:        {is_char}')
    print(f"is above ascii: {is_national}")
    print(f"entry lengths:  {entry_lengths}")
    print(f"max length:     {max_length}")
    print(f"fixed length:   {is_fixed_length}")


table = "olist_geolocation_dataset.csv"
column = "geolocation_state"
describe_table(current_table=table, current_column=column)

current column: geolocation_state

59763     SP
906649    SC
374958    SP
13924     SP
837660    MS
Name: geolocation_state, dtype: object

dtype:  object
table:  olist_geolocation_dataset.csv
column: geolocation_state

contains NULL:  False
is above ascii: False
entry lengths:  [2]
max length:     2
fixed length:   True


In [195]:
df = dfs[table]
print(df[column].unique())

print(df[df[column] == 20])

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
                               order_id  order_item_id  \
11951  1b15974a0141d54e36626dca3fdc731a             20   
57316  8272b63d03f5f79c56e9e4120aec44ef             20   
75122  ab14fdcfbe524636d65ee38360e22ce8             20   

                             product_id                         seller_id  \
11951  ee3d532c8a438679776d222e997606b3  8e6d7754bc7e0f22c96d255ebda59eba   
57316  270516a3f41dc035aa87d220228f844c  2709af9587499e95e803a6498a5a56e9   
75122  9571759451b1d780ee7c15012ea109d4  ce27a3cc3c8cc1ea79d11e561e9bebb6   

       shipping_limit_date  price  freight_value  
11951  2018-03-01 02:50:48  100.0          10.12  
57316  2017-07-21 18:25:23    1.2           7.89  
75122  2017-08-30 14:30:23   98.7          14.44  
